# Block 2 — merge: verify, coverage, pair completeness, `K_FINAL`

CPU only. Runs after both candidate-generation notebooks have marked their stages done. It does not
copy or rewrite candidate data — the country folders stay the storage. It:

1. checks every country's `TRAIN` (and `TEST`) marker,
2. verifies every committed shard (file present, row count matches its `done-*.json`),
3. checks coverage — every S1 queried exactly once, with the right role and country,
4. computes pair completeness per country x role from the positives sidecars,
5. picks `K_FINAL` (smallest K whose recall on roles the bi-encoder never saw hits the target),
6. writes `workspace/block2/prod/merged/manifest.json` — **the single entry point Blocks 3 and 4
   read** through `b2p.iter_candidates(...)` — plus `pc_report.json`.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import numpy as np
import pandas as pd
import b2_common as b2
import b2_prod as b2p

In [ ]:
# Config
SMOKE = False               # must match what the producers ran with
INCLUDE_TEST = True         # False = merge train only
TARGET_RECALL = 0.995       # K_FINAL target on TREE-FIT/TREE-DEV/CALIBRATION/HOLDOUT
K_FINAL_OVERRIDE = None     # set an int to force K_FINAL
b2p.use_smoke_root(SMOKE)

In [ ]:
# Stage 0 — every producer stage must be committed
train_countries, train_null = b2p.countries_in("train")
test_countries, test_null = b2p.countries_in("test")
need = [(c, "TRAIN") for c in train_countries]
if INCLUDE_TEST:
    need += [(c, "TEST") for c in test_countries]
print("required markers:", need)
b2p.require_markers(need)

## 1 — verify committed shards

In [ ]:
# Stage 1 — verify
shards = [s for c in b2p.produced_countries() for s in b2p.collect_shards(c)]
if not INCLUDE_TEST:
    shards = [s for s in shards if s["split"] == "train"]
print(f"{len(shards)} shards across {sorted({s['country'] for s in shards})}")
problems = b2p.verify_shards(shards)
for p in problems:
    print("PROBLEM:", p)
assert not problems, "fix the shards above (re-run that role in its producer notebook)"
fps = {(s["country"], s.get("checkpoint"), s.get("checkpoint_fingerprint")) for s in shards if s["n_rows"]}
print("checkpoints used:", sorted(fps, key=str))

## 2 — coverage: every S1 queried exactly once, right role, right country

In [ ]:
# Stage 2 — coverage
produced = b2p.produced_queries(shards)
coverage = {}
for split in (["train", "test"] if INCLUDE_TEST else ["train"]):
    exp = b2p.expected_queries(split)
    rep = b2p.coverage_report(exp, produced[produced["split"] == split], k_needed=b2p.DEFAULT_K_FINAL)
    coverage[split] = rep
    print(f"\n--- {split} ---")
    for k, v in rep.items():
        print(f"  {k}: {v}")

if not SMOKE:
    for split, rep in coverage.items():
        assert rep["n_duplicated_across_shards"] == 0 and rep["n_extra"] == 0, split
        assert rep["n_role_mismatch"] == 0 and rep["n_country_mismatch"] == 0, split
        if rep["n_missing"]:
            print(f"WARNING {split}: {rep['n_missing']:,} S1 never queried -> "
                  f"{rep['missing_by_country_role']}  (NULL-country rows show as country None)")
else:
    print("\nSMOKE run: missing queries are expected (3,000 per role).")

## 3 — pair completeness (train roles)

In [ ]:
# Stage 3 — pair completeness
pp, roles_df = b2.load_labels()
bucket_map = dict(zip(b2._get_s1_id_series(roles_df), roles_df["match_bucket"].astype(str)))
pos = b2p.load_pos_sidecars([s for s in shards if s["split"] == "train"])
pc_report = {"per_country_role": {}, "per_role": {}, "unbiased_pooled": None}
for (c, role), sub in pos.groupby(["country", "role"]):
    r = b2p.pc_from_positives(sub, bucket_map=bucket_map)
    pc_report["per_country_role"][f"{c}/{role}"] = r
    b2p.print_pc(r, f"{c} {role}")
for role, sub in pos.groupby("role"):
    pc_report["per_role"][role] = b2p.pc_from_positives(sub, bucket_map=bucket_map)
unb = pos[pos["role"].isin(b2p.UNBIASED_ROLES)]
pc_report["unbiased_pooled"] = b2p.pc_from_positives(unb, bucket_map=bucket_map)
b2p.print_pc(pc_report["unbiased_pooled"], "POOLED — roles the bi-encoder never saw")
print("\nCE-FIT is optimistic by construction (the bi-encoder trained on 90% of it).")
print("HOLDOUT pair completeness is the recall ceiling: no later block can recover a pair missed here.")

## 4 — choose `K_FINAL` (recall vs downstream volume)

In [ ]:
# Stage 4 — K_FINAL
k_sugg, r_sugg = b2p.suggest_k_final(pos, target=TARGET_RECALL)
print(f"suggested K_FINAL = {k_sugg}  (pooled unbiased pair recall {r_sugg * 100:.2f}%, "
      f"target {TARGET_RECALL * 100:.2f}%)")

# downstream volume: every Block 3 feature row / Block 4 cross-encoder pass is one candidate row
downstream_roles = b2p.UNBIASED_ROLES + ["CE-DEV"]
vol = produced[(produced["split"] == "test") | produced["role"].isin(downstream_roles)]
print(f"\n{'K':>4} | {'unbiased pair recall':>20} | {'rows for Block 3/4 (TREE-*, CAL, HOLDOUT, CE-DEV, TEST)':>55}")
for k in b2p.EVAL_K_LIST:
    r = ((unb["rank_found"] > 0) & (unb["rank_found"] <= k)).mean()
    rows = int(np.minimum(vol["n_cands"].to_numpy(), k).sum())
    print(f"{k:>4} | {r * 100:19.2f}% | {rows:>55,}")

K_FINAL = K_FINAL_OVERRIDE or k_sugg
print(f"\nK_FINAL = {K_FINAL}")

## 5 — write the merged manifest (the Block 3 / Block 4 entry point)

In [ ]:
# Stage 5 — merged manifest
checkpoints = {}
for c in b2p.produced_countries():
    fp = {s.get("checkpoint_fingerprint") for s in shards if s["country"] == c and s["n_rows"]}
    checkpoints[c] = {"checkpoint": b2p.checkpoint_key_for(c), "fingerprints": sorted(x for x in fp if x)}
b2p.write_merged_manifest(shards, K_FINAL, pc_report, coverage, checkpoints)
with open(f"{b2p.merged_root()}/_DONE_MERGED", "w") as f:
    f.write(str(K_FINAL))

m = b2p.load_merged_manifest()
print(f"manifest: {len(m['shards'])} shards, K_GEN={m['k_gen']}, K_FINAL={m['k_final']}")

## 6 — test-side sanity (no labels): top-1 score distribution per country

In [ ]:
# Stage 6 — score distribution
rows = []
for s in shards:
    if s.get("top1_score_pcts"):
        rows.append({"country": s["country"], "split": s["split"], "role": s["role"],
                     "median_top1": s["top1_score_pcts"][2], "p5_top1": s["top1_score_pcts"][0],
                     "n_queries": s["n_queries"]})
dist = (pd.DataFrame(rows).groupby(["country", "split"])
        .apply(lambda g: pd.Series({
            "median_top1": np.average(g["median_top1"], weights=g["n_queries"]),
            "p5_top1": np.average(g["p5_top1"], weights=g["n_queries"]),
            "n_queries": g["n_queries"].sum()}), include_groups=False))
print(dist.round(4))
print("\nCountries without labels (FRANCE, ...) are only as trustworthy as this comparison.")
print("\nBlock 2 done. Next: 03_block3.ipynb")